In [ ]:
# Install dependencies (RunPod / Colab)
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'transformers', 'datasets', 'scikit-learn',
    'matplotlib', 'seaborn', 'pandas', 'numpy',
], check=True)
print('All packages ready.')

In [ ]:
# Download SST-2 from HuggingFace and save locally
import json, os
from datasets import load_dataset

os.makedirs('SST-2', exist_ok=True)
print('Downloading glue/sst2 ...')
raw = load_dataset('glue', 'sst2')
for hf_split, local_name in [('train','train'),('validation','validation'),('test','test')]:
    out_path = os.path.join('SST-2', f'{local_name}.json')
    if os.path.exists(out_path):
        print(f'  {local_name}.json already exists, skipping.')
        continue
    records = []
    for row in raw[hf_split]:
        records.append({
            'sentence_original':     row['sentence'],
            'sentence_preprocessed': row['sentence'],
            'label':                 row['label'],
        })
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f'  Saved {local_name}.json  ({len(records)} samples)')
print('Data ready.')

In [ ]:
import json, os, random, time, shutil
import numpy as np
import pandas as pd
from datetime import datetime

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, RobertaModel
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)
from sklearn.manifold import TSNE

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

# â”€â”€ Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
SEED          = 42
DATASET       = 'SST-2'
MODEL_NAME    = 'roberta-base'
NUM_EPOCHS    = 3
BATCH_SIZE    = 32
LEARNING_RATE = 2e-5
MAX_LENGTH    = 128
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.06
PROJ_DIM      = 128
TEMPERATURE   = 0.3
LAMBDA_SCL    = 0.5
TEXT_COLUMN   = 'sentence_original'
LABEL_COLUMN  = 'label'
ROOT_DIR      = os.getcwd()

set_seed(SEED)
sns.set_style('whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Model:  {MODEL_NAME}  |  Epochs: {NUM_EPOCHS}  |  BS: {BATCH_SIZE}  |  LR: {LEARNING_RATE}')
print(f'Temp:   {TEMPERATURE}  |  Lambda SCL: {LAMBDA_SCL}  |  Proj dim: {PROJ_DIM}')

In [ ]:
dataset_path = os.path.join(ROOT_DIR, DATASET)

def load_split(split_name):
    fp = os.path.join(dataset_path, f'{split_name}.json')
    if not os.path.exists(fp):
        raise FileNotFoundError(f'{fp} not found â€” run cell-00-data first.')
    with open(fp, 'r', encoding='utf-8') as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    print(f'  {split_name}: {len(df)} samples')
    return df

print(f'Loading data from: {dataset_path}')
df_train      = load_split('train')
df_validation = load_split('validation')
df_test       = load_split('test')
print(df_train[['sentence_original', 'label']].head(3).to_string(index=False))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SSTDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts  = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc  = tokenizer(self.texts[idx], max_length=MAX_LENGTH,
                         padding='max_length', truncation=True, return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_texts  = df_train[TEXT_COLUMN].fillna('').tolist()
train_labels = df_train[LABEL_COLUMN].tolist()
val_texts    = df_validation[TEXT_COLUMN].fillna('').tolist()
val_labels   = df_validation[LABEL_COLUMN].tolist()
test_texts   = df_test[TEXT_COLUMN].fillna('').tolist()

train_dataset = SSTDataset(train_texts, train_labels)
val_dataset   = SSTDataset(val_texts,   val_labels)
test_dataset  = SSTDataset(test_texts,  None)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')
print(f'Total steps: {total_steps} | Warmup steps: {warmup_steps}')

In [ ]:
class SupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.3):
        super().__init__()
        self.temperature = temperature

    def forward(self, embeddings, labels):
        sim_matrix = torch.matmul(embeddings, embeddings.T) / self.temperature
        labels_row = labels.unsqueeze(0)
        mask       = torch.eq(labels_row, labels_row.T).float()
        B          = embeddings.shape[0]
        self_mask  = torch.eye(B, device=embeddings.device).bool()
        mask       = mask.masked_fill(self_mask, 0)
        sim_matrix = sim_matrix.masked_fill(self_mask, float('-inf'))
        sim_max, _ = sim_matrix.max(dim=1, keepdim=True)
        sim_matrix = sim_matrix - sim_max.detach()
        exp_sim    = torch.exp(sim_matrix).masked_fill(self_mask, 0)
        log_prob   = sim_matrix - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)
        pos_count  = mask.sum(dim=1)
        mean_lp    = (mask * log_prob).sum(dim=1) / (pos_count + 1e-8)
        valid      = pos_count > 0
        if valid.sum() == 0:
            return torch.tensor(0.0, device=embeddings.device, requires_grad=True)
        return -mean_lp[valid].mean()


class RoBERTaForSCL(nn.Module):
    def __init__(self, model_name, num_labels, proj_dim=128):
        super().__init__()
        self.roberta         = RobertaModel.from_pretrained(model_name)
        hidden               = self.roberta.config.hidden_size
        self.classifier      = nn.Sequential(nn.Dropout(0.1), nn.Linear(hidden, num_labels))
        self.projection_head = nn.Sequential(
            nn.Linear(hidden, proj_dim), nn.ReLU(), nn.Linear(proj_dim, proj_dim)
        )

    def forward(self, input_ids, attention_mask, return_projected=False):
        out     = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = out.last_hidden_state[:, 0, :]
        logits  = self.classifier(cls_emb)
        if return_projected:
            proj = F.normalize(self.projection_head(cls_emb), p=2, dim=1)
            return logits, proj, cls_emb
        return logits, cls_emb

    def get_cls_embeddings(self, input_ids, attention_mask):
        with torch.no_grad():
            out = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state[:, 0, :]

print('Model classes defined.')

In [ ]:
print('Training CE baseline ...')
set_seed(SEED)

model_ce   = RoBERTaForSCL(MODEL_NAME, num_labels=2, proj_dim=PROJ_DIM).to(device)
opt_ce     = torch.optim.AdamW(model_ce.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
sch_ce     = get_linear_schedule_with_warmup(opt_ce, warmup_steps, total_steps)
scaler_ce  = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

ce_train_losses, ce_val_losses = [], []
ce_train_accs,   ce_val_accs   = [], []

start_ce = time.time()
for epoch in range(1, NUM_EPOCHS + 1):
    model_ce.train()
    ep_loss, preds_all, labs_all = 0.0, [], []
    for batch in train_loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labs = batch['labels'].to(device)
        opt_ce.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            logits, _ = model_ce(ids, mask)
            loss      = F.cross_entropy(logits, labs)
        scaler_ce.scale(loss).backward()
        scaler_ce.unscale_(opt_ce)
        torch.nn.utils.clip_grad_norm_(model_ce.parameters(), 1.0)
        scaler_ce.step(opt_ce); scaler_ce.update(); sch_ce.step()
        ep_loss += loss.item()
        preds_all.extend(torch.argmax(logits, 1).cpu().numpy())
        labs_all.extend(labs.cpu().numpy())

    model_ce.eval()
    vl, vp, vl_ = 0.0, [], []
    with torch.no_grad():
        for batch in val_loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labs = batch['labels'].to(device)
            logits, _ = model_ce(ids, mask)
            vl += F.cross_entropy(logits, labs).item()
            vp.extend(torch.argmax(logits, 1).cpu().numpy())
            vl_.extend(labs.cpu().numpy())

    tl = ep_loss/len(train_loader); vll = vl/len(val_loader)
    ta = accuracy_score(labs_all, preds_all); va = accuracy_score(vl_, vp)
    ce_train_losses.append(tl); ce_val_losses.append(vll)
    ce_train_accs.append(ta);   ce_val_accs.append(va)
    print(f'  Epoch {epoch}/{NUM_EPOCHS} | Train Loss: {tl:.4f} | Val Loss: {vll:.4f} | Train Acc: {ta:.4f} | Val Acc: {va:.4f}')

train_time_ce = time.time() - start_ce
print(f'CE done in {train_time_ce:.1f}s  |  Final Val Acc: {ce_val_accs[-1]:.4f}')

In [ ]:
print('Training SCL model ...')
set_seed(SEED)

model_scl  = RoBERTaForSCL(MODEL_NAME, num_labels=2, proj_dim=PROJ_DIM).to(device)
scl_crit   = SupervisedContrastiveLoss(TEMPERATURE)
opt_scl    = torch.optim.AdamW(model_scl.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
sch_scl    = get_linear_schedule_with_warmup(opt_scl, warmup_steps, total_steps)
scaler_scl = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

scl_train_losses_ce, scl_val_losses_ce   = [], []
scl_train_losses_scl                     = []
scl_train_accs, scl_val_accs             = [], []

start_scl = time.time()
for epoch in range(1, NUM_EPOCHS + 1):
    model_scl.train()
    ep_ce, ep_scl, preds_all, labs_all = 0.0, 0.0, [], []
    for batch in train_loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labs = batch['labels'].to(device)
        opt_scl.zero_grad()
        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            logits, proj, _ = model_scl(ids, mask, return_projected=True)
            lce  = F.cross_entropy(logits, labs)
            lscl = scl_crit(proj, labs)
            loss = lce + LAMBDA_SCL * lscl
        scaler_scl.scale(loss).backward()
        scaler_scl.unscale_(opt_scl)
        torch.nn.utils.clip_grad_norm_(model_scl.parameters(), 1.0)
        scaler_scl.step(opt_scl); scaler_scl.update(); sch_scl.step()
        ep_ce += lce.item(); ep_scl += lscl.item()
        preds_all.extend(torch.argmax(logits, 1).cpu().numpy())
        labs_all.extend(labs.cpu().numpy())

    model_scl.eval()
    vce, vp, vl_ = 0.0, [], []
    with torch.no_grad():
        for batch in val_loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labs = batch['labels'].to(device)
            logits, _ = model_scl(ids, mask)
            vce += F.cross_entropy(logits, labs).item()
            vp.extend(torch.argmax(logits, 1).cpu().numpy())
            vl_.extend(labs.cpu().numpy())

    tce = ep_ce/len(train_loader); tscl = ep_scl/len(train_loader); vll = vce/len(val_loader)
    ta  = accuracy_score(labs_all, preds_all); va = accuracy_score(vl_, vp)
    scl_train_losses_ce.append(tce); scl_val_losses_ce.append(vll)
    scl_train_losses_scl.append(tscl)
    scl_train_accs.append(ta); scl_val_accs.append(va)
    print(f'  Epoch {epoch}/{NUM_EPOCHS} | CE: {tce:.4f} | SCL: {tscl:.4f} | Val CE: {vll:.4f} | Train Acc: {ta:.4f} | Val Acc: {va:.4f}')

train_time_scl = time.time() - start_scl
print(f'SCL done in {train_time_scl:.1f}s  |  Final Val Acc: {scl_val_accs[-1]:.4f}')

In [ ]:
def evaluate_model(model, loader, has_labels=True):
    model.eval()
    preds, true_labs, total_loss = [], [], 0.0
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            logits, _ = model(ids, mask)
            if has_labels and 'labels' in batch:
                labs = batch['labels'].to(device)
                total_loss += F.cross_entropy(logits, labs).item()
                true_labs.extend(labs.cpu().numpy())
            preds.extend(torch.argmax(logits, 1).cpu().numpy())
    avg_loss = total_loss / len(loader) if true_labs else None
    return np.array(preds), np.array(true_labs) if true_labs else None, avg_loss

def compute_metrics(y_true, y_pred, loss=None):
    d = {
        'accuracy':            round(float(accuracy_score(y_true, y_pred)), 4),
        'precision_macro':     round(float(precision_score(y_true, y_pred, average='macro', zero_division=0)), 4),
        'recall_macro':        round(float(recall_score(y_true, y_pred, average='macro', zero_division=0)), 4),
        'f1_macro':            round(float(f1_score(y_true, y_pred, average='macro', zero_division=0)), 4),
        'precision_per_class': [round(float(x),4) for x in precision_score(y_true, y_pred, average=None, zero_division=0)],
        'recall_per_class':    [round(float(x),4) for x in recall_score(y_true, y_pred, average=None, zero_division=0)],
        'f1_per_class':        [round(float(x),4) for x in f1_score(y_true, y_pred, average=None, zero_division=0)],
        'confusion_matrix':    confusion_matrix(y_true, y_pred).tolist(),
    }
    if loss is not None:
        d['loss'] = round(float(loss), 4)
    return d

y_tr_pred_ce,  y_tr_true_ce,  tr_loss_ce  = evaluate_model(model_ce,  train_loader)
y_val_pred_ce, y_val_true_ce, val_loss_ce  = evaluate_model(model_ce,  val_loader)
y_test_pred_ce, _, _                        = evaluate_model(model_ce,  test_loader, has_labels=False)

y_tr_pred_scl,  y_tr_true_scl,  tr_loss_scl  = evaluate_model(model_scl, train_loader)
y_val_pred_scl, y_val_true_scl, val_loss_scl  = evaluate_model(model_scl, val_loader)
y_test_pred_scl, _, _                          = evaluate_model(model_scl, test_loader, has_labels=False)

ce_train_res  = compute_metrics(y_tr_true_ce,  y_tr_pred_ce,  tr_loss_ce)
ce_val_res    = compute_metrics(y_val_true_ce, y_val_pred_ce, val_loss_ce)
scl_train_res = compute_metrics(y_tr_true_scl, y_tr_pred_scl, tr_loss_scl)
scl_val_res   = compute_metrics(y_val_true_scl,y_val_pred_scl,val_loss_scl)
class_labels  = sorted(np.unique(y_val_true_ce).tolist())

print('=== CE  | Val Acc: {:.4f} | F1: {:.4f}'.format(ce_val_res['accuracy'],  ce_val_res['f1_macro']))
print('=== SCL | Val Acc: {:.4f} | F1: {:.4f}'.format(scl_val_res['accuracy'], scl_val_res['f1_macro']))
print()
print('Classification Report (CE - Validation):')
print(classification_report(y_val_true_ce, y_val_pred_ce, zero_division=0))
print('Classification Report (SCL - Validation):')
print(classification_report(y_val_true_scl, y_val_pred_scl, zero_division=0))

In [ ]:
epochs_range = list(range(1, NUM_EPOCHS + 1))

fig_cm_ce, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(np.array(ce_val_res['confusion_matrix']), annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('RoBERTa-CE  -  SST-2', fontweight='bold')
fig_cm_ce.tight_layout()

fig_cm_scl, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(np.array(scl_val_res['confusion_matrix']), annot=True, fmt='d', cmap='Greens',
            xticklabels=class_labels, yticklabels=class_labels, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('RoBERTa-SCL  -  SST-2', fontweight='bold')
fig_cm_scl.tight_layout()

fig_acc_ce, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs_range, ce_train_accs, label='Train', linewidth=2, color='#1f77b4')
ax.plot(epochs_range, ce_val_accs,   label='Val',   linewidth=2, color='#ff7f0e')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('RoBERTa-CE  -  SST-2', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3); fig_acc_ce.tight_layout()

fig_acc_scl, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs_range, scl_train_accs, label='Train', linewidth=2, color='#1f77b4')
ax.plot(epochs_range, scl_val_accs,   label='Val',   linewidth=2, color='#ff7f0e')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('RoBERTa-SCL  -  SST-2', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3); fig_acc_scl.tight_layout()

fig_loss_ce, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs_range, ce_train_losses, label='Train Loss', linewidth=2, color='#1f77b4')
ax.plot(epochs_range, ce_val_losses,   label='Val Loss',   linewidth=2, color='#ff7f0e')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss (CE)')
ax.set_title('RoBERTa-CE  -  SST-2', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3); fig_loss_ce.tight_layout()

fig_loss_scl, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs_range, scl_train_losses_ce,  label='Train CE',  linewidth=2, color='#1f77b4')
ax.plot(epochs_range, scl_val_losses_ce,    label='Val CE',    linewidth=2, color='#ff7f0e')
ax.plot(epochs_range, scl_train_losses_scl, label='Train SCL', linewidth=2, color='#2ca02c', linestyle='--')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('RoBERTa-SCL  -  SST-2 (CE + SCL losses)', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3); fig_loss_scl.tight_layout()

fig_compare, ax = plt.subplots(figsize=(10, 5))
ax.plot(epochs_range, ce_val_accs,  label='CE  Val Acc',  linewidth=2, color='#1f77b4')
ax.plot(epochs_range, scl_val_accs, label='SCL Val Acc',  linewidth=2, color='#d62728')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('CE vs SCL  -  Validation Accuracy  -  SST-2', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3); fig_compare.tight_layout()

print('All figures created.')

In [ ]:
def save_results(model_obj, figures,
                 train_results, val_results, test_results,
                 train_time, entries_per_second,
                 dataset, model_name, model_hf_name,
                 num_epochs, batch_size, learning_rate, max_length,
                 text_column, label_column, class_labels,
                 df_train, df_validation, df_test, root_dir,
                 extra_params=None, optimizer_info=None, scheduler_info=None):

    model_size_bytes = sum(p.numel() * p.element_size() for p in model_obj.parameters())
    ts       = datetime.now().strftime('%d_%m_%Y_%H_%M_%S')
    run_name = f'{model_hf_name}_ep{num_epochs}_bs{batch_size}_lr{learning_rate}_{ts}'
    out_dir  = os.path.join(root_dir, model_name, dataset, run_name)
    os.makedirs(out_dir, exist_ok=True)

    for fig_name, fig_obj in figures.items():
        fig_obj.savefig(os.path.join(out_dir, fig_name), dpi=150, bbox_inches='tight')

    total_p     = sum(p.numel() for p in model_obj.parameters())
    trainable_p = sum(p.numel() for p in model_obj.parameters() if p.requires_grad)

    metrics = {
        'model':          model_name,
        'model_hf_name':  model_hf_name,
        'dataset':        dataset,
        'run_name':       run_name,
        'timestamp':      ts,
        'seed':           SEED,
        'model_params': {
            'num_epochs': num_epochs, 'batch_size': batch_size,
            'learning_rate': learning_rate, 'max_length': max_length,
            'total_params': total_p, 'trainable_params': trainable_p,
            **(extra_params or {}),
        },
        'model_size':   {'bytes': model_size_bytes, 'MB': round(model_size_bytes/(1024*1024), 4)},
        'training': {
            'training_time_seconds': round(train_time, 4),
            'entries_per_second':    round(entries_per_second, 2),
            'num_epochs':            num_epochs,
            'train_samples':         len(df_train),
        },
        'data': {
            'train_samples':      len(df_train),
            'validation_samples': len(df_validation),
            'test_samples':       len(df_test) if df_test is not None else 0,
            'text_column':        text_column,
            'label_column':       label_column,
            'classes':            class_labels,
        },
        'optimizer':          optimizer_info or {},
        'scheduler':          scheduler_info or {},
        'train_results':      train_results,
        'validation_results': val_results,
    }
    if test_results is not None:
        metrics['test_results'] = test_results

    mp = os.path.join(out_dir, 'metrics.json')
    with open(mp, 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    print(f'Saved: {out_dir}')
    return out_dir, mp, run_name


opt_info = {'name': 'AdamW', 'lr': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY}
sch_info = {'name': 'linear_warmup', 'warmup_ratio': WARMUP_RATIO, 'total_steps': total_steps}

output_dir_ce, _, run_name_ce = save_results(
    model_obj=model_ce,
    figures={
        'confusion_matrix.png':   fig_cm_ce,
        'accuracy_curves.png':    fig_acc_ce,
        'loss_curves.png':        fig_loss_ce,
        'ce_vs_scl_compare.png':  fig_compare,
    },
    train_results=ce_train_res, val_results=ce_val_res, test_results=None,
    train_time=train_time_ce,
    entries_per_second=(len(train_labels)*NUM_EPOCHS)/train_time_ce,
    dataset=DATASET, model_name='RoBERTa-CE', model_hf_name=MODEL_NAME,
    num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, max_length=MAX_LENGTH,
    text_column=TEXT_COLUMN, label_column=LABEL_COLUMN, class_labels=class_labels,
    df_train=df_train, df_validation=df_validation, df_test=df_test, root_dir=ROOT_DIR,
    extra_params={'lambda_scl': 0.0},
    optimizer_info=opt_info, scheduler_info=sch_info,
)

output_dir_scl, _, run_name_scl = save_results(
    model_obj=model_scl,
    figures={
        'confusion_matrix.png':   fig_cm_scl,
        'accuracy_curves.png':    fig_acc_scl,
        'loss_curves.png':        fig_loss_scl,
        'ce_vs_scl_compare.png':  fig_compare,
    },
    train_results=scl_train_res, val_results=scl_val_res, test_results=None,
    train_time=train_time_scl,
    entries_per_second=(len(train_labels)*NUM_EPOCHS)/train_time_scl,
    dataset=DATASET, model_name='RoBERTa-SCL', model_hf_name=MODEL_NAME,
    num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, max_length=MAX_LENGTH,
    text_column=TEXT_COLUMN, label_column=LABEL_COLUMN, class_labels=class_labels,
    df_train=df_train, df_validation=df_validation, df_test=df_test, root_dir=ROOT_DIR,
    extra_params={'temperature': TEMPERATURE, 'lambda_scl': LAMBDA_SCL, 'proj_dim': PROJ_DIM},
    optimizer_info=opt_info, scheduler_info=sch_info,
)

print()
print('=== FINAL RESULTS ===')
print(f'  CE  | Val Acc: {ce_val_res["accuracy"]:.4f} | F1: {ce_val_res["f1_macro"]:.4f}')
print(f'  SCL | Val Acc: {scl_val_res["accuracy"]:.4f} | F1: {scl_val_res["f1_macro"]:.4f}')

In [ ]:
print('Extracting CLS embeddings for t-SNE (val set) ...')

def get_cls_embs(model, loader):
    model.eval()
    embs, labs = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            e    = model.get_cls_embeddings(ids, mask)
            embs.append(e.cpu().numpy())
            if 'labels' in batch:
                labs.extend(batch['labels'].numpy())
    return np.vstack(embs), np.array(labs)

emb_ce,  lab_ce  = get_cls_embs(model_ce,  val_loader)
emb_scl, lab_scl = get_cls_embs(model_scl, val_loader)

print('Running t-SNE ...')
tsne_model = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=1000)
coords_ce  = tsne_model.fit_transform(emb_ce)
coords_scl = tsne_model.fit_transform(emb_scl)

colors = {0: '#d62728', 1: '#1f77b4'}
fig_tsne, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, coords, labs, title in [
        (axes[0], coords_ce,  lab_ce,  'RoBERTa-CE'),
        (axes[1], coords_scl, lab_scl, 'RoBERTa-SCL'),
]:
    for cls_id, color in colors.items():
        m = labs == cls_id
        ax.scatter(coords[m, 0], coords[m, 1], c=color, alpha=0.6, s=10,
                   label='Positive' if cls_id == 1 else 'Negative')
    ax.set_title(f'{title}  -  t-SNE  (val set)', fontweight='bold')
    ax.legend(markerscale=3); ax.axis('off')
fig_tsne.tight_layout()
fig_tsne.savefig(os.path.join(output_dir_ce,  'tsne.png'), dpi=150, bbox_inches='tight')
fig_tsne.savefig(os.path.join(output_dir_scl, 'tsne.png'), dpi=150, bbox_inches='tight')
plt.show()
print('t-SNE saved.')

In [ ]:
from sklearn.model_selection import train_test_split

FRACTIONS = [0.01, 0.05, 0.10, 0.50, 1.00]
FS_SEEDS  = [42, 43, 44]

def train_eval_quick(train_df, val_df, use_scl, seed, n_epochs=NUM_EPOCHS):
    set_seed(seed)
    tr_ds = SSTDataset(train_df[TEXT_COLUMN].fillna('').tolist(), train_df[LABEL_COLUMN].tolist())
    vl_ds = SSTDataset(val_df[TEXT_COLUMN].fillna('').tolist(),   val_df[LABEL_COLUMN].tolist())
    tr_ld = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
    vl_ld = DataLoader(vl_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    ts_   = len(tr_ld) * n_epochs; ws_ = int(ts_ * WARMUP_RATIO)
    m     = RoBERTaForSCL(MODEL_NAME, num_labels=2, proj_dim=PROJ_DIM).to(device)
    opt_  = torch.optim.AdamW(m.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    sch_  = get_linear_schedule_with_warmup(opt_, ws_, ts_)
    sc_   = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
    crit_ = SupervisedContrastiveLoss(TEMPERATURE)

    for _ in range(n_epochs):
        m.train()
        for batch in tr_ld:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labs = batch['labels'].to(device)
            opt_.zero_grad()
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                if use_scl:
                    logits, proj, _ = m(ids, mask, return_projected=True)
                    loss = F.cross_entropy(logits, labs) + LAMBDA_SCL * crit_(proj, labs)
                else:
                    logits, _ = m(ids, mask)
                    loss = F.cross_entropy(logits, labs)
            sc_.scale(loss).backward()
            sc_.unscale_(opt_); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            sc_.step(opt_); sc_.update(); sch_.step()

    m.eval()
    preds, tlabs = [], []
    with torch.no_grad():
        for batch in vl_ld:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            logits, _ = m(ids, mask)
            preds.extend(torch.argmax(logits, 1).cpu().numpy())
            tlabs.extend(batch['labels'].numpy())
    return accuracy_score(tlabs, preds)

print('Running few-shot experiment ...')
fs_results = {f: {'ce': [], 'scl': []} for f in FRACTIONS}
for frac in FRACTIONS:
    for seed in FS_SEEDS:
        if frac < 1.0:
            subset, _ = train_test_split(df_train, train_size=frac, random_state=seed,
                                         stratify=df_train[LABEL_COLUMN])
        else:
            subset = df_train
        acc_ce  = train_eval_quick(subset, df_validation, use_scl=False, seed=seed)
        acc_scl = train_eval_quick(subset, df_validation, use_scl=True,  seed=seed)
        fs_results[frac]['ce'].append(acc_ce); fs_results[frac]['scl'].append(acc_scl)
        print(f'  frac={frac:.0%}  n={len(subset):5d}  seed={seed}  CE={acc_ce:.4f}  SCL={acc_scl:.4f}')

# Plot
fig_fs, ax = plt.subplots(figsize=(10, 6))
fracs_pct  = [f*100 for f in FRACTIONS]
ce_m  = [np.mean(fs_results[f]['ce'])  for f in FRACTIONS]
scl_m = [np.mean(fs_results[f]['scl']) for f in FRACTIONS]
ce_s  = [np.std(fs_results[f]['ce'])   for f in FRACTIONS]
scl_s = [np.std(fs_results[f]['scl'])  for f in FRACTIONS]
ax.errorbar(fracs_pct, ce_m,  yerr=ce_s,  label='CE',  fmt='-o', linewidth=2, capsize=4, color='#1f77b4')
ax.errorbar(fracs_pct, scl_m, yerr=scl_s, label='SCL', fmt='-s', linewidth=2, capsize=4, color='#d62728')
ax.set_xlabel('Training data (%)'); ax.set_ylabel('Val Accuracy')
ax.set_title('Few-shot Learning  -  CE vs SCL  -  SST-2', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3); fig_fs.tight_layout()
fig_fs.savefig(os.path.join(output_dir_ce,  'fewshot.png'), dpi=150, bbox_inches='tight')
fig_fs.savefig(os.path.join(output_dir_scl, 'fewshot.png'), dpi=150, bbox_inches='tight')

pd.DataFrame([{'fraction':f,'n_train':int(f*len(df_train)),
               'ce_mean':round(np.mean(fs_results[f]['ce']),4),'ce_std':round(np.std(fs_results[f]['ce']),4),
               'scl_mean':round(np.mean(fs_results[f]['scl']),4),'scl_std':round(np.std(fs_results[f]['scl']),4)}
              for f in FRACTIONS]).to_csv(os.path.join(output_dir_scl, 'fewshot_results.csv'), index=False)
print('Few-shot done.')

In [ ]:
NOISE_LEVELS = [0.0, 0.05, 0.10, 0.15]

def add_label_noise(df, rate, seed):
    df2  = df.copy()
    rng  = np.random.RandomState(seed)
    idx  = rng.choice(len(df2), size=int(rate*len(df2)), replace=False)
    col  = df2.columns.get_loc(LABEL_COLUMN)
    df2.iloc[idx, col] = 1 - df2.iloc[idx][LABEL_COLUMN]
    return df2

print('Running noise robustness experiment ...')
noise_rows = []
for nl in NOISE_LEVELS:
    noisy = add_label_noise(df_train, nl, SEED)
    acc_ce  = train_eval_quick(noisy, df_validation, use_scl=False, seed=SEED)
    acc_scl = train_eval_quick(noisy, df_validation, use_scl=True,  seed=SEED)
    noise_rows.append({'noise_rate': nl, 'ce_acc': round(acc_ce,4), 'scl_acc': round(acc_scl,4)})
    print(f'  noise={nl:.0%}  CE={acc_ce:.4f}  SCL={acc_scl:.4f}')

fig_noise, ax = plt.subplots(figsize=(9, 5))
noise_pct = [r['noise_rate']*100 for r in noise_rows]
ax.plot(noise_pct, [r['ce_acc']  for r in noise_rows], '-o', label='CE',  linewidth=2, color='#1f77b4')
ax.plot(noise_pct, [r['scl_acc'] for r in noise_rows], '-s', label='SCL', linewidth=2, color='#d62728')
ax.set_xlabel('Label noise (%)'); ax.set_ylabel('Val Accuracy')
ax.set_title('Noise Robustness  -  CE vs SCL  -  SST-2', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3); fig_noise.tight_layout()
fig_noise.savefig(os.path.join(output_dir_ce,  'noise_robustness.png'), dpi=150, bbox_inches='tight')
fig_noise.savefig(os.path.join(output_dir_scl, 'noise_robustness.png'), dpi=150, bbox_inches='tight')

pd.DataFrame(noise_rows).to_csv(os.path.join(output_dir_scl, 'noise_results.csv'), index=False)
print('Noise robustness done.')

In [ ]:
TEMPS   = [0.1, 0.3, 0.5]
LAMBDAS = [0.1, 0.5, 1.0]

print('Running hyperparameter ablation (temperature x lambda) ...')
abl_rows = []
for temp in TEMPS:
    for lam in LAMBDAS:
        set_seed(SEED)
        m_   = RoBERTaForSCL(MODEL_NAME, num_labels=2, proj_dim=PROJ_DIM).to(device)
        ts_  = len(train_loader) * NUM_EPOCHS; ws_ = int(ts_ * WARMUP_RATIO)
        opt_ = torch.optim.AdamW(m_.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        sch_ = get_linear_schedule_with_warmup(opt_, ws_, ts_)
        sc_  = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
        cr_  = SupervisedContrastiveLoss(temp)
        for _ in range(NUM_EPOCHS):
            m_.train()
            for batch in train_loader:
                ids  = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                labs = batch['labels'].to(device)
                opt_.zero_grad()
                with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                    logits, proj, _ = m_(ids, mask, return_projected=True)
                    loss = F.cross_entropy(logits, labs) + lam * cr_(proj, labs)
                sc_.scale(loss).backward(); sc_.unscale_(opt_)
                torch.nn.utils.clip_grad_norm_(m_.parameters(), 1.0)
                sc_.step(opt_); sc_.update(); sch_.step()
        m_.eval()
        preds, tlabs = [], []
        with torch.no_grad():
            for batch in val_loader:
                ids  = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                logits, _ = m_(ids, mask)
                preds.extend(torch.argmax(logits, 1).cpu().numpy())
                tlabs.extend(batch['labels'].numpy())
        acc = accuracy_score(tlabs, preds)
        abl_rows.append({'temperature': temp, 'lambda': lam, 'val_acc': round(acc, 4)})
        print(f'  tau={temp}  lambda={lam}  acc={acc:.4f}')

abl_df = pd.DataFrame(abl_rows)
abl_df.to_csv(os.path.join(output_dir_scl, 'ablation_results.csv'), index=False)
print(abl_df.pivot(index='temperature', columns='lambda', values='val_acc').to_string())

In [ ]:
# Save model weights into their respective result folders
# State dict saves the full model (backbone + classifier + projection head)

models_dir_ce  = os.path.join(output_dir_ce,  'model')
models_dir_scl = os.path.join(output_dir_scl, 'model')
os.makedirs(models_dir_ce,  exist_ok=True)
os.makedirs(models_dir_scl, exist_ok=True)

# Save state dicts
torch.save(model_ce.state_dict(),  os.path.join(models_dir_ce,  'model_ce.pt'))
torch.save(model_scl.state_dict(), os.path.join(models_dir_scl, 'model_scl.pt'))

# Save tokenizer alongside (needed for inference)
tokenizer.save_pretrained(models_dir_ce)
tokenizer.save_pretrained(models_dir_scl)

# Save model config so you know how to reload
reload_info = {
    'model_class':  'RoBERTaForSCL',
    'model_name':   MODEL_NAME,
    'num_labels':   2,
    'proj_dim':     PROJ_DIM,
    'state_dict':   'model/model_ce.pt',   # relative to output_dir
    'tokenizer':    'model/',
    'reload_code': (
        "model = RoBERTaForSCL('roberta-base', num_labels=2, proj_dim=128)
"
        "model.load_state_dict(torch.load('model_ce.pt', map_location='cpu'))
"
        "model.eval()"
    ),
}
with open(os.path.join(models_dir_ce, 'reload_info.json'), 'w') as f:
    json.dump(reload_info, f, indent=2)

reload_info['state_dict'] = 'model/model_scl.pt'
with open(os.path.join(models_dir_scl, 'reload_info.json'), 'w') as f:
    json.dump(reload_info, f, indent=2)

for d, name in [(models_dir_ce, 'CE'), (models_dir_scl, 'SCL')]:
    size_mb = sum(os.path.getsize(os.path.join(d, f)) for f in os.listdir(d)) / (1024*1024)
    print(f'  {name} model saved to: {d}  ({size_mb:.0f} MB)')

print()
print('To reload CE model later:')
print('  model = RoBERTaForSCL("roberta-base", num_labels=2, proj_dim=128)')
print('  model.load_state_dict(torch.load("model_ce.pt", map_location="cpu"))')
print('  model.eval()')

In [ ]:
# GLUE submission TSV files (official SST-2 test labels are hidden)
glue_ce  = pd.DataFrame({'index': range(len(y_test_pred_ce)),  'prediction': y_test_pred_ce})
glue_scl = pd.DataFrame({'index': range(len(y_test_pred_scl)), 'prediction': y_test_pred_scl})

tsv_ce  = os.path.join(output_dir_ce,  'SST-2_CE.tsv')
tsv_scl = os.path.join(output_dir_scl, 'SST-2_SCL.tsv')
glue_ce.to_csv(tsv_ce,  sep='\t', index=False)
glue_scl.to_csv(tsv_scl, sep='\t', index=False)

print(f'GLUE CE  submission ({len(glue_ce)} rows):  {tsv_ce}')
print(f'GLUE SCL submission ({len(glue_scl)} rows): {tsv_scl}')
print()
print('First 5 CE predictions:')
print(glue_ce.head().to_string(index=False))

In [ ]:
# Zip results for RunPod file-browser download
zip_ce  = shutil.make_archive('results_RoBERTa-CE',  'zip', output_dir_ce)
zip_scl = shutil.make_archive('results_RoBERTa-SCL', 'zip', output_dir_scl)

print('Download these files from the RunPod file browser:')
for z in [zip_ce, zip_scl]:
    size_mb = os.path.getsize(z) / (1024 * 1024)
    print(f'  {z}  ({size_mb:.1f} MB)')